In [1]:
landing_path = (
    "abfss://Fleet_Logistics_Engineering@onelake.dfs.fabric.microsoft.com/"
    "Fleet_Logistics_Lakehouse.Lakehouse/Files/Landing/safety_incidents.csv"
)

StatementMeta(, e72579e3-3444-4a08-98c9-3cf797ca274b, 3, Finished, Available, Finished, False)

In [2]:
df_safety = (
    spark.read
    .option("header", "true")
    .csv(landing_path)
)

display(df_safety.limit(20))

df_safety.printSchema()

print(f"Source records: {df_safety.count()}")

StatementMeta(, e72579e3-3444-4a08-98c9-3cf797ca274b, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d66387d3-916e-448a-825c-06f8e280d8dd)

root
 |-- incident_id: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- truck_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- incident_date: string (nullable = true)
 |-- incident_type: string (nullable = true)
 |-- location_city: string (nullable = true)
 |-- location_state: string (nullable = true)
 |-- at_fault_flag: string (nullable = true)
 |-- injury_flag: string (nullable = true)
 |-- vehicle_damage_cost: string (nullable = true)
 |-- cargo_damage_cost: string (nullable = true)
 |-- claim_amount: string (nullable = true)
 |-- preventable_flag: string (nullable = true)
 |-- description: string (nullable = true)

Source records: 170


In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    DateType
)

# ============================================================
# SAFETY INCIDENTS — EXPLICIT SCHEMA
# ============================================================

safety_schema = StructType([
    StructField("incident_id", StringType(), True),
    StructField("trip_id", StringType(), True),
    StructField("truck_id", StringType(), True),
    StructField("driver_id", StringType(), True),
    StructField("incident_date", DateType(), True),
    StructField("incident_type", StringType(), True),
    StructField("location_city", StringType(), True),
    StructField("location_state", StringType(), True),
    StructField("at_fault_flag", StringType(), True),
    StructField("injury_flag", StringType(), True),
    StructField("vehicle_damage_cost", DoubleType(), True),
    StructField("cargo_damage_cost", DoubleType(), True),
    StructField("claim_amount", DoubleType(), True),
    StructField("preventable_flag", StringType(), True),
    StructField("description", StringType(), True)
])

landing_path = (
    "abfss://Fleet_Logistics_Engineering@onelake.dfs.fabric.microsoft.com/"
    "Fleet_Logistics_Lakehouse.Lakehouse/Files/Landing/safety_incidents.csv"
)

df_safety = (
    spark.read
    .option("header", "true")
    .schema(safety_schema)
    .csv(landing_path)
)

display(df_safety.limit(20))

df_safety.printSchema()

print(f"Source records: {df_safety.count()}")

StatementMeta(, ea47c13f-02de-4276-8871-9cce00610a0b, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 795a7db7-44c5-4af8-b149-2f58104c1e29)

root
 |-- incident_id: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- truck_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- incident_date: date (nullable = true)
 |-- incident_type: string (nullable = true)
 |-- location_city: string (nullable = true)
 |-- location_state: string (nullable = true)
 |-- at_fault_flag: string (nullable = true)
 |-- injury_flag: string (nullable = true)
 |-- vehicle_damage_cost: double (nullable = true)
 |-- cargo_damage_cost: double (nullable = true)
 |-- claim_amount: double (nullable = true)
 |-- preventable_flag: string (nullable = true)
 |-- description: string (nullable = true)

Source records: 170


In [2]:
# ============================================================
# DATA QUALITY VALIDATION
# ============================================================

# 1. NULL primary key
null_incident_ids = (
    df_safety
    .filter(F.col("incident_id").isNull())
    .count()
)

print(f"NULL incident IDs: {null_incident_ids}")


# 2. Duplicate primary key
duplicate_incident_ids = (
    df_safety
    .groupBy("incident_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate incident IDs: {duplicate_incident_ids}")


# 3. NULL incident dates
null_incident_dates = (
    df_safety
    .filter(F.col("incident_date").isNull())
    .count()
)

print(f"NULL incident dates: {null_incident_dates}")


# 4. Invalid vehicle damage cost
invalid_vehicle_damage = (
    df_safety
    .filter(
        F.col("vehicle_damage_cost").isNull() |
        (F.col("vehicle_damage_cost") < 0)
    )
    .count()
)

print(f"Invalid vehicle damage costs: {invalid_vehicle_damage}")


# 5. Invalid cargo damage cost
invalid_cargo_damage = (
    df_safety
    .filter(
        F.col("cargo_damage_cost").isNull() |
        (F.col("cargo_damage_cost") < 0)
    )
    .count()
)

print(f"Invalid cargo damage costs: {invalid_cargo_damage}")


# 6. Invalid claim amount
invalid_claim_amount = (
    df_safety
    .filter(
        F.col("claim_amount").isNull() |
        (F.col("claim_amount") < 0)
    )
    .count()
)

print(f"Invalid claim amounts: {invalid_claim_amount}")

StatementMeta(, ea47c13f-02de-4276-8871-9cce00610a0b, 4, Finished, Available, Finished, False)

NULL incident IDs: 0
Duplicate incident IDs: 0
NULL incident dates: 0
Invalid vehicle damage costs: 0
Invalid cargo damage costs: 0
Invalid claim amounts: 0


In [3]:
# ============================================================
# REFERENTIAL INTEGRITY — TRIP
# ============================================================

invalid_safety_trip_ids = (
    df_safety
    .filter(F.col("trip_id").isNotNull())
    .join(
        spark.table("bronze_trips").select("trip_id"),
        on="trip_id",
        how="left_anti"
    )
    .count()
)

print(f"Invalid non-NULL trip IDs: {invalid_safety_trip_ids}")

StatementMeta(, ea47c13f-02de-4276-8871-9cce00610a0b, 5, Finished, Available, Finished, False)

Invalid non-NULL trip IDs: 0


In [4]:
# ============================================================
# REFERENTIAL INTEGRITY — TRUCK
# ============================================================

invalid_safety_truck_ids = (
    df_safety
    .filter(F.col("truck_id").isNotNull())
    .join(
        spark.table("bronze_trucks").select("truck_id"),
        on="truck_id",
        how="left_anti"
    )
    .count()
)

print(f"Invalid non-NULL truck IDs: {invalid_safety_truck_ids}")

StatementMeta(, ea47c13f-02de-4276-8871-9cce00610a0b, 6, Finished, Available, Finished, False)

Invalid non-NULL truck IDs: 0


In [5]:
# ============================================================
# REFERENTIAL INTEGRITY — DRIVER
# ============================================================

invalid_safety_driver_ids = (
    df_safety
    .filter(F.col("driver_id").isNotNull())
    .join(
        spark.table("bronze_drivers").select("driver_id"),
        on="driver_id",
        how="left_anti"
    )
    .count()
)

print(f"Invalid non-NULL driver IDs: {invalid_safety_driver_ids}")

StatementMeta(, ea47c13f-02de-4276-8871-9cce00610a0b, 7, Finished, Available, Finished, False)

Invalid non-NULL driver IDs: 0


In [6]:
# ============================================================
# FINAL VALIDATION
# ============================================================

if null_incident_ids > 0:
    raise ValueError(
        "ETL failed: NULL incident_id values detected."
    )

if duplicate_incident_ids > 0:
    raise ValueError(
        "ETL failed: Duplicate incident_id values detected."
    )

if null_incident_dates > 0:
    raise ValueError(
        "ETL failed: NULL incident_date values detected."
    )

if invalid_vehicle_damage > 0:
    raise ValueError(
        "ETL failed: Invalid vehicle damage costs detected."
    )

if invalid_cargo_damage > 0:
    raise ValueError(
        "ETL failed: Invalid cargo damage costs detected."
    )

if invalid_claim_amount > 0:
    raise ValueError(
        "ETL failed: Invalid claim amounts detected."
    )

if invalid_safety_trip_ids > 0:
    raise ValueError(
        "ETL failed: Safety incidents contain "
        "trip IDs not found in bronze_trips."
    )

if invalid_safety_truck_ids > 0:
    raise ValueError(
        "ETL failed: Safety incidents contain "
        "truck IDs not found in bronze_trucks."
    )

if invalid_safety_driver_ids > 0:
    raise ValueError(
        "ETL failed: Safety incidents contain "
        "driver IDs not found in bronze_drivers."
    )

print(
    "Safety Incidents data quality and "
    "referential-integrity validation passed."
)

StatementMeta(, ea47c13f-02de-4276-8871-9cce00610a0b, 8, Finished, Available, Finished, False)

Safety Incidents data quality and referential-integrity validation passed.


In [7]:
# ============================================================
# ADD ETL METADATA
# ============================================================

df_safety_bronze = (
    df_safety
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.lit("safety_incidents.csv"))
)

display(df_safety_bronze.limit(10))

StatementMeta(, ea47c13f-02de-4276-8871-9cce00610a0b, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, db40c3d7-dc39-47f4-a867-d861c3776e93)

In [8]:
df_safety_bronze.createOrReplaceTempView(
    "safety_incidents_source"
)

StatementMeta(, ea47c13f-02de-4276-8871-9cce00610a0b, 10, Finished, Available, Finished, False)

In [9]:
# ============================================================
# CREATE BRONZE TABLE
# ============================================================

spark.sql("""
CREATE TABLE IF NOT EXISTS bronze_safety_incidents (
    incident_id STRING,
    trip_id STRING,
    truck_id STRING,
    driver_id STRING,
    incident_date DATE,
    incident_type STRING,
    location_city STRING,
    location_state STRING,
    at_fault_flag STRING,
    injury_flag STRING,
    vehicle_damage_cost DOUBLE,
    cargo_damage_cost DOUBLE,
    claim_amount DOUBLE,
    preventable_flag STRING,
    description STRING,
    ingestion_timestamp TIMESTAMP,
    source_file STRING
)
""")

print("bronze_safety_incidents table is ready.")

StatementMeta(, ea47c13f-02de-4276-8871-9cce00610a0b, 11, Finished, Available, Finished, False)

bronze_safety_incidents table is ready.


In [10]:
# ============================================================
# SAFETY INCIDENTS BRONZE MERGE
# ============================================================

spark.sql("""
MERGE INTO bronze_safety_incidents AS target

USING safety_incidents_source AS source

ON target.incident_id = source.incident_id

WHEN MATCHED THEN
    UPDATE SET
        target.trip_id = source.trip_id,
        target.truck_id = source.truck_id,
        target.driver_id = source.driver_id,
        target.incident_date = source.incident_date,
        target.incident_type = source.incident_type,
        target.location_city = source.location_city,
        target.location_state = source.location_state,
        target.at_fault_flag = source.at_fault_flag,
        target.injury_flag = source.injury_flag,
        target.vehicle_damage_cost = source.vehicle_damage_cost,
        target.cargo_damage_cost = source.cargo_damage_cost,
        target.claim_amount = source.claim_amount,
        target.preventable_flag = source.preventable_flag,
        target.description = source.description,
        target.ingestion_timestamp = source.ingestion_timestamp,
        target.source_file = source.source_file

WHEN NOT MATCHED THEN
    INSERT (
        incident_id,
        trip_id,
        truck_id,
        driver_id,
        incident_date,
        incident_type,
        location_city,
        location_state,
        at_fault_flag,
        injury_flag,
        vehicle_damage_cost,
        cargo_damage_cost,
        claim_amount,
        preventable_flag,
        description,
        ingestion_timestamp,
        source_file
    )

    VALUES (
        source.incident_id,
        source.trip_id,
        source.truck_id,
        source.driver_id,
        source.incident_date,
        source.incident_type,
        source.location_city,
        source.location_state,
        source.at_fault_flag,
        source.injury_flag,
        source.vehicle_damage_cost,
        source.cargo_damage_cost,
        source.claim_amount,
        source.preventable_flag,
        source.description,
        source.ingestion_timestamp,
        source.source_file
    )
""")

print("Safety Incidents Bronze MERGE completed successfully.")

StatementMeta(, ea47c13f-02de-4276-8871-9cce00610a0b, 12, Finished, Available, Finished, False)

Safety Incidents Bronze MERGE completed successfully.


In [11]:
bronze_safety_count = (
    spark.table("bronze_safety_incidents")
    .count()
)

print(f"Bronze safety incident records: {bronze_safety_count}")

StatementMeta(, ea47c13f-02de-4276-8871-9cce00610a0b, 13, Finished, Available, Finished, False)

Bronze safety incident records: 170
